# Batch Scoring: New Clients

Automated scoring pipeline for new, unlabelled clients:

1. Picks up new client CSVs (`inference_*.csv`) dropped in the shared UC volume `/Volumes/workspace/default/churn_raw` — ingests them into the raw table (skipping clients already known) and records which file each client came from
   - Only clients that have **never been scored** are scored on each run
2. Applies **the same** feature engineering as training — reads `02_etl_and_features/sql/feature_store.sql` directly, so this notebook can never drift from the training pipeline
3. Scores with the **champion** model alias (`@champion` — promoting a new champion requires no edit here)
4. Applies the deploy cut-off (0.29, best F1) and **appends** the full row + scores to the output table with a `scored_at` timestamp and the `source_file` each client came from

Designed to run unattended in a Lakeflow Job. All five parameters below can be overridden per run by the job definition.

In [0]:
%pip install feature-engine lightgbm

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 46.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 121.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [feature-engine]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
from datetime import datetime

# --- Job-overridable parameters ---
dbutils.widgets.text("source_table", "workspace.default.inference", "Raw clients table")
dbutils.widgets.text("model_uri", "models:/workspace.default.churn_lightgbm_balanced_tuned@champion", "Champion model URI")
dbutils.widgets.text("threshold", "0.29", "Deploy cut-off (best F1)")
dbutils.widgets.text("output_table", "workspace.default.churn_scores", "Scored output table (append)")
dbutils.widgets.text("ingest_folder", "/Volumes/workspace/default/churn_raw", "UC volume where new client CSVs (inference_*.csv) are dropped — shared across the team, watched by the job trigger")

SOURCE_TABLE = dbutils.widgets.get("source_table")
MODEL_URI    = dbutils.widgets.get("model_uri")
THRESHOLD    = float(dbutils.widgets.get("threshold"))
OUTPUT_TABLE = dbutils.widgets.get("output_table")
# Resolve the (possibly relative) path — so any override style works
INGEST_FOLDER = os.path.abspath(dbutils.widgets.get("ingest_folder"))

# Timestamp of this scoring round — separates rounds in the append-only output table
SCORED_AT = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"source   : {SOURCE_TABLE}")
print(f"model    : {MODEL_URI}")
print(f"threshold: {THRESHOLD}")
print(f"output   : {OUTPUT_TABLE} (append, round {SCORED_AT})")
print(f"watching : {INGEST_FOLDER}")

source   : workspace.default.inference
model    : models:/workspace.default.churn_lightgbm_balanced_tuned@champion
threshold: 0.29
output   : workspace.default.churn_scores (append, round 2026-11-12 17:31:15)


In [0]:
import fnmatch
import os
from pyspark.sql import functions as F

# --- 1. Discover new client files dropped in the ingest folder (naming convention: inference_*.csv) ---
assert os.path.isdir(INGEST_FOLDER), f"Ingest folder not found: {INGEST_FOLDER} — check the ingest_folder parameter"
new_files = sorted(
    f for f in os.listdir(INGEST_FOLDER) if fnmatch.fnmatch(f.lower(), "inference_*.csv")
)

print(f"New client files found: {new_files or 'none'}")

# --- 2. Raw table columns; stamped columns are produced by this pipeline, not by incoming files ---
raw_tbl = spark.table(SOURCE_TABLE)
raw_schema = raw_tbl.schema
STAMPED_COLS = ("source_file", "ingested_at")
data_cols = [c for c in raw_schema.fieldNames() if c not in STAMPED_COLS]

# --- 3. Ingest client VERSIONS never seen before — whole-line dedup, not per customerID ---
# A client whose line changed (e.g. added a product) is a NEW version: ingest and re-score it.
# Only exact duplicates of an already-ingested line are skipped (exceptAll = null-safe comparison).
seen = raw_tbl.select(data_cols)
ingest_parts = []
for fname in new_files:
    df_csv = (spark.read
        .option("header", True)
        .option("multiLine", True)    # CustomerFeedback contains newlines inside quoted fields
        .option("escape", '"')        # ...and embedded double quotes (escaped "" per RFC 4180)
        .option("nullValue", "null")  # source files encode missing values as the literal string 'null'
        .csv(f"{INGEST_FOLDER}/{fname}"))
    missing = [c for c in data_cols if c not in df_csv.columns]
    assert not missing, f"{fname} is missing columns vs {SOURCE_TABLE}: {missing}"
    typed = df_csv.select([F.col(c).cast(raw_schema[c].dataType).alias(c) for c in data_cols])

    fresh = typed.exceptAll(seen)
    n_fresh = fresh.count()
    if n_fresh > 0:
        ingest_parts.append(fresh
            .withColumn("source_file", F.lit(fname))        # stamped the moment the file is ingested
            .withColumn("ingested_at", F.current_timestamp()))
        seen = seen.union(fresh)  # the same version in a later file this round is not re-ingested
    print(f"{fname}: {n_fresh} new client versions")

df_new_versions = None
for part in ingest_parts:
    df_new_versions = part if df_new_versions is None else df_new_versions.unionByName(part)

n_new = df_new_versions.count() if df_new_versions is not None else 0
if n_new > 0:
    (df_new_versions
        .write.format("delta").mode("append").saveAsTable(SOURCE_TABLE))
    print(f"Ingested {n_new} new client versions into {SOURCE_TABLE}")
else:
    print("No new client versions — nothing ingested.")

In [0]:
# Run the EXACT same feature engineering used at training time.
# feature_store.sql contains a {dataset} placeholder in the FROM clause.
FEATURE_SQL_PATH = "../../02_etl_and_features/sql/feature_store.sql"

with open(FEATURE_SQL_PATH) as f:
    feature_query = f.read().format(dataset="inference")

# Score client versions whose FEATURE LINE is not in the output table yet: a brand-new client,
# or an existing client whose data changed (so their feature line differs). Exact repeats are
# skipped — dedup on the whole line, not just the client ID.
df_features_all = spark.sql(feature_query)
scored_features = spark.table(OUTPUT_TABLE).select(df_features_all.columns)
df_features = df_features_all.exceptAll(scored_features)

n_clients = df_features.count()
if n_clients == 0:
    print("No new or changed clients to score — nothing to do this round.")
    dbutils.notebook.exit("No new or changed clients to score.")
print(f"New / changed client versions to score: {n_clients}")

# File map: for each client to score, the source_file of its LATEST ingested version —
# stamped the moment its file arrived, so scoring never re-derives it.
from pyspark.sql import Window

latest_version = (spark.table(SOURCE_TABLE)
    .filter("source_file is not null and ingested_at is not null")
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("customerID").orderBy(F.col("ingested_at").desc())))
    .filter("_rn = 1")
    .select("customerID", "source_file"))
file_map_pd = (latest_version
    .join(df_features.select("customerID"), "customerID", "left_semi")
    .toPandas())
print(f"File map entries this round: {len(file_map_pd)}")

display(df_features.limit(5))

New clients to score: 1409


customerID,isMale,SeniorCitizen,isPartner,HasDependents,MonthlyIncome,nbrTenure,tenure_calc,ContractTier,isContractLong,isContractOneYear,isContractTwoYear,isPaperlessBilling,isPaymentMethodAuto,isPaymentMethodCheck,isPaymentMethod_BankTransfer,isPaymentMethod_CreditCard,isPaymentMethod_ElectronicCheck,isPaymentMethod_MailedCheck,hasPhoneService,hasMultipleLines,hasInternetService,isInternetService_DSL,isInternetService_Fiber,hasOnlineSecurity,hasOnlineBackup,hasDeviceProtection,hasTechSupport,hasStreamingTV,hasStreamingMovies,MonthlyCharges,TotalCharges,isNewCustomer,tenureGroup,nbrInternetServicesAddOnQty,nbrInternetServicesAddOnQtyPct,hasSecurityBundle,hasStreamingBundle,nbrStreamingCount,ServiceTier,TotalSpend,TotalSpendPct,isM2M_dual,isM2M_internetOnly,isOneYear_dual,isTwoYear_dual,chargeRatio,numServices,chargesPerService
4226-KKDON,1,0,0,0,1361,8,8.0,M2M,0,0,0,1,0,1,0,0,1,0,1,0,1,1,0,0,1,0,1,1,0,66.65,520.95,0,2_7to12,3,0.500000,0,0,1,dual,533.2,0.9770255063765941,1,0,0,0,0.04897134459955915,5,13.330000000000002
8838-GPHZP,0,0,0,0,3850,63,63.0,two-year,1,0,1,1,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,20.6,1298.7,0,5_49plus,0,0.000000,0,0,0,phone_only,1297.8000000000002,1.0006934812760055,0,0,0,0,0.005350649350649351,1,20.6
0320-JDNQG,1,0,1,1,2736,23,23.0,M2M,0,0,0,1,0,1,0,0,1,0,1,0,1,0,1,1,0,0,1,1,1,99.85,2331.3,0,3_13to24,4,0.666667,0,1,2,dual,2296.5499999999997,1.015131392741286,1,0,0,0,0.03649488304093567,6,16.641666666666666
4837-PZTIC,0,0,0,0,1139,47,null,two-year,1,0,1,0,0,1,0,0,0,1,0,0,1,1,0,1,0,1,1,0,0,41.9,null,0,4_25to48,3,0.500000,0,0,0,internet_only,1969.3,null,0,0,0,0,0.03678665496049166,4,10.475
7748-UMTRK,0,1,0,1,5211,30,29.0,M2M,0,0,0,1,0,1,0,0,1,0,1,1,1,0,1,0,0,0,1,1,1,101.3,2974.5,0,4_25to48,3,0.500000,0,1,2,dual,3039.0,0.9787759131293189,1,0,0,0,0.019439646900786797,6,16.883333333333333


In [0]:
import mlflow

model = mlflow.sklearn.load_model(MODEL_URI)

X = df_features.toPandas()

# Train/serve consistency: the model must see exactly the features it was trained on
missing = [c for c in model.feature_names_in_ if c not in X.columns]
assert not missing, f"Feature columns missing vs champion model: {missing}"
assert list(model.classes_) == [0, 1], model.classes_  # class 1 (churn) must be predict_proba column 1

X["ProbaChurn"] = model.predict_proba(X[model.feature_names_in_])[:, 1]
X["prediction"] = (X["ProbaChurn"] >= THRESHOLD).map({True: "Yes", False: "No"})
X["scored_at"] = SCORED_AT

X[["customerID", "ProbaChurn", "prediction"]].head(10)

,customerID,ProbaChurn,prediction
0,4226-KKDON,0.203938,No
1,8838-GPHZP,0.030046,No
2,0320-JDNQG,0.305393,Yes
3,4837-PZTIC,0.049472,No
4,7748-UMTRK,0.489411,Yes
5,0823-HSCDJ,0.294949,Yes
6,1085-LDWAM,0.050034,No
7,2786-GCDPI,0.352860,Yes
8,4001-TSBTV,0.323322,Yes
9,5196-WPYOW,0.057985,No


In [0]:
# Source file each client came from; 'backfill' when the client has no file record
if not file_map_pd.empty:
    X = X.merge(file_map_pd, on="customerID", how="left")
if "source_file" in X.columns:
    X["source_file"] = X["source_file"].fillna("backfill")
else:
    X["source_file"] = "backfill"  # no new ingest this round (e.g. clients ingested by an earlier round that crashed before scoring) — no file record, label as backfill

# Full audit row: every engineered feature the model saw + the 4 scoring columns
output_cols = list(df_features.columns) + ["ProbaChurn", "prediction", "scored_at", "source_file"]

spark_df = spark.createDataFrame(X[output_cols])

(spark_df.write
   .format("delta")
   .option("mergeSchema", "true")  # churn_scores already exists without source_file — column is added safely
   .mode("append")                 # append-only: every scoring round is preserved
   .saveAsTable(OUTPUT_TABLE))

print(f"Appended {spark_df.count()} scored clients to {OUTPUT_TABLE} (round {SCORED_AT})")

Appended 1409 scored clients to workspace.default.churn_scores (round 2026-11-12 17:31:15)


In [0]:
round_df = spark.sql(f"SELECT * FROM {OUTPUT_TABLE} WHERE scored_at = '{SCORED_AT}'")

n_scored = round_df.count()
n_yes    = round_df.filter("prediction = 'Yes'").count()
rev_risk = round_df.filter("prediction = 'Yes'").agg({"MonthlyCharges": "sum"}).first()[0] or 0

assert n_scored == n_clients, f"Round row count mismatch: scored {n_scored} vs input {n_clients}"

print(f"Scored this round   : {n_scored}")
print(f"Predicted churners  : {n_yes} ({n_yes / n_scored:.1%})")
print(f"Monthly rev at risk : ${rev_risk:,.2f}")
print(f"\nClients by source file this round:")
display(round_df.groupBy("source_file").count())

print(f"\nTop 10 highest-risk new clients:")
display(round_df.select("customerID", "ProbaChurn", "prediction", "ContractTier", "tenureGroup", "ServiceTier", "MonthlyCharges").orderBy("ProbaChurn", ascending=False).limit(10))

Scored this round   : 1409
Predicted churners  : 549 (39.0%)
Monthly rev at risk : $40,900.85

Top 10 highest-risk new clients:


customerID,ProbaChurn,prediction,ContractTier,tenureGroup,ServiceTier,MonthlyCharges
6567-HOOPW,0.8141853958219958,Yes,M2M,1_0to6,dual,79.2
4910-GMJOT,0.8047399064784083,Yes,M2M,1_0to6,dual,94.6
5919-TMRGD,0.8030467712710704,Yes,M2M,1_0to6,dual,79.35
0107-YHINA,0.7992089971224342,Yes,M2M,1_0to6,dual,99.75
3097-NQYSN,0.7990167954947397,Yes,M2M,1_0to6,dual,86.85
9300-AGZNL,0.7937662018267119,Yes,M2M,1_0to6,dual,94.0
4929-XIHVW,0.7903474743784002,Yes,M2M,1_0to6,dual,95.5
0318-QUUOB,0.787072257634943,Yes,M2M,1_0to6,dual,80.55
0259-GBZSH,0.7857581923475246,Yes,M2M,1_0to6,dual,85.65
2027-FECZV,0.7842880275977456,Yes,M2M,2_7to12,dual,106.7
